In [2]:
%pip install -q -r ../requirements.txt

  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [94 lines of output]
      Rust not found, installing into a temporary directory
      Python reports SOABI: cpython-314-x86_64-linux-gnu
      Computed rustc target triple: x86_64-unknown-linux-gnu
      Installation directory: /home/codespace/.cache/puccinialin
      Rustup already downloaded
      Installing rust to /home/codespace/.cache/puccinialin/rustup
      warn: it looks like you have an existing rustup settings file at:
      warn: /home/codespace/.cache/puccinialin/rustup/settings.toml
      info: profile set to minimal
      info: setting default host tuple to x86_64-unknown-linux-gnu
      warn: Updating existing toolchain, profile choice will be ignored
      info: syncing channel updates for stable-x86_64-unknown-linux-gnu
      info: default toolchain set to stable-x86_64-unknown-linux-gnu
      Checking if cargo is installed
    

In [ ]:
import json, os, sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from tqdm.auto import tqdm
sys.path.append('../src')
from load_data import load_splits
from evaluate import evaluate, path_baseline, save_preds


In [ ]:
SMOKE, OVERFIT = False, False
SEED, BATCH_SIZE, LR, WEIGHT_DECAY = 42, 256, 1e-3, 1e-4
MAX_EPOCHS = 100 if OVERFIT else 1 if SMOKE else 20
EVAL_EVERY, PATIENCE, MIN_DELTA = (10 if SMOKE or OVERFIT else 500), 3, 1e-4
TAG = 'bilstm_overfit' if OVERFIT else 'bilstm_smoke' if SMOKE else 'bilstm'
RESULTS, CKPT, LOG = '../results', f'../checkpoints/{TAG}_model2.pt', f'../results/{TAG}_train_log.csv'
os.makedirs(RESULTS, exist_ok=True); os.makedirs('../checkpoints', exist_ok=True)
torch.manual_seed(SEED); np.random.seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
use_amp = device == 'cuda'
print('device:', device, '| torch', torch.__version__)

## Shared data and character encoding
The cleaning notebook produced a 67-entry vocabulary (`<PAD>`, `<UNK>`, and 65 retained characters) and `max_len=256`.

In [1]:
splits = load_splits()
if SMOKE: splits = {s: d.sample(2000 if s == 'train' else 500, random_state=SEED) for s, d in splits.items()}
if OVERFIT: splits['train'] = splits['val'] = splits['train'].groupby('label').head(3)
with open('../data/vocab.json') as f: vocab = json.load(f)
with open('../data/config.json') as f: config = json.load(f)
char_to_idx = {c: i for i, c in enumerate(vocab['index_to_string'])}
PAD_IDX, UNK_IDX, MAX_LEN = vocab['pad_idx'], vocab['unk_idx'], config['max_len']
print({s: len(d) for s, d in splits.items()}); print('vocab:', len(char_to_idx), 'MAX_LEN:', MAX_LEN)

NameError: name 'load_splits' is not defined

In [ ]:
class URLDataset(Dataset):
    def __init__(self, df):
        self.urls, self.labels = df.URL.tolist(), df.label.astype('int64').tolist()
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        ids = [char_to_idx.get(c, UNK_IDX) for c in self.urls[i][:MAX_LEN]]
        return torch.tensor(ids, dtype=torch.long), self.labels[i]

def collate(batch):
    ids, labels = zip(*batch)
    lengths = torch.tensor([len(x) for x in ids], dtype=torch.long)
    ids = torch.nn.utils.rnn.pad_sequence(ids, batch_first=True, padding_value=PAD_IDX)
    return ids, lengths, torch.tensor(labels, dtype=torch.long)

datasets = {s: URLDataset(d) for s, d in splits.items()}
train_loader = DataLoader(datasets['train'], batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
val_loader = DataLoader(datasets['val'], batch_size=512, collate_fn=collate)
test_loader = DataLoader(datasets['test'], batch_size=512, collate_fn=collate)

## BiLSTM classifier
Packing ignores padding. The final forward and backward hidden states are concatenated. Class 0 remains phishing, so softmax column 0 is `p_phish`.

In [ ]:
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim=64, hidden_dim=128, num_layers=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD_IDX)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=num_layers, batch_first=True, bidirectional=True, dropout=dropout)
        self.dropout, self.fc = nn.Dropout(dropout), nn.Linear(hidden_dim * 2, 2)
    def forward(self, ids, lengths):
        packed = pack_padded_sequence(self.embedding(ids), lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (hidden, _) = self.lstm(packed)
        return self.fc(self.dropout(torch.cat((hidden[-2], hidden[-1]), dim=1)))

model = BiLSTMClassifier(len(char_to_idx)).to(device)
print(f'parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
loss_fn = nn.CrossEntropyLoss()
@torch.no_grad()
def run_eval(loader):
    model.eval(); total, probs = 0.0, []
    for ids, lengths, y in loader:
        ids, lengths, y = ids.to(device), lengths.to(device), y.to(device)
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=use_amp): logits = model(ids, lengths).float()
        total += loss_fn(logits, y).item() * len(y); probs.append(torch.softmax(logits, -1)[:, 0].cpu())
    return total / len(loader.dataset), torch.cat(probs).numpy()

def predict(urls):
    ds = URLDataset(pd.DataFrame({'URL': list(urls), 'label': 1}))
    return run_eval(DataLoader(ds, batch_size=512, collate_fn=collate))[1]

## Training, validation, and early stopping

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps, warmup = MAX_EPOCHS * len(train_loader), int(0.06 * MAX_EPOCHS * len(train_loader))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda s: s / max(1, warmup) if s < warmup else max(0., (total_steps - s) / max(1, total_steps - warmup)))
scaler, writer = torch.amp.GradScaler('cuda', enabled=use_amp), SummaryWriter(f'../runs/{TAG}')
log, running, best_val, bad, step, stopped = [], [], float('inf'), 0, 0, False
for epoch in range(1, MAX_EPOCHS + 1):
    for ids, lengths, y in tqdm(train_loader, desc=f'epoch {epoch}'):
        model.train(); ids, lengths, y = ids.to(device), lengths.to(device), y.to(device)
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=use_amp): loss = loss_fn(model(ids, lengths), y)
        optimizer.zero_grad(); scaler.scale(loss).backward(); scaler.unscale_(optimizer); nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(optimizer); scaler.update(); scheduler.step()
        step += 1; running.append(loss.item()); writer.add_scalar('loss/train_step', running[-1], step)
        if step % EVAL_EVERY == 0:
            val_loss, p_val = run_eval(val_loader); m = evaluate(datasets['val'].labels, p_val); improved = val_loss < best_val - MIN_DELTA
            if improved: best_val, bad = val_loss, 0; torch.save(model.state_dict(), CKPT)
            else: bad += 1
            row = {'step': step, 'epoch': epoch, 'train_loss': np.mean(running), 'val_loss': val_loss, 'val_acc': m['accuracy'], 'val_recall': m['recall'], 'val_f1': m['f1'], 'lr': scheduler.get_last_lr()[0], 'best': improved}
            log.append(row); running = []; pd.DataFrame(log).to_csv(LOG, index=False); writer.add_scalars('loss', {'train': row['train_loss'], 'val': val_loss}, step)
            print(f"step {step} | train {row['train_loss']:.4f} | val {val_loss:.4f} | f1 {m['f1']:.4f}" + (' | best' if improved else ''))
            if bad >= PATIENCE: print(f'early stopping: val loss did not improve for {PATIENCE} evals'); stopped = True; break
    if stopped: break
writer.close()

## Test and save comparable predictions

In [ ]:
model.load_state_dict(torch.load(CKPT, map_location=device, weights_only=True))
test = splits['test']; test_loss, p_test = run_eval(test_loader); test_metrics = evaluate(test.label, p_test)
save_preds(f'{RESULTS}/{TAG}_test_original.csv', test.URL, test.label, p_test)
with open(f'{RESULTS}/{TAG}_test_original.json', 'w') as f: json.dump(test_metrics, f, indent=1)
comparison = pd.DataFrame({'BiLSTM': test_metrics, 'path rule': evaluate(test.label, path_baseline(test.URL))}).drop('confusion_matrix').T
display(comparison.astype(float).round(4))